# SQL Window Aggregates

Companion to the [complete article](https://todatabeyond.com/blog/introduction-to-sql-window-functions-simplifying-complex-queries-for-efficient-data-analysis). Run in order with Python 3.11+, SQLite 3.25+ and Jupyter/Colab. Standard-library sqlite3 executes real SQL in isolated in-memory databases. Setup registers the small %%sql helper explicitly; no hidden SQL extension, extra data-library install, server, keys, files, GPU or dataset downloads.

All ten original input rows are transcribed from the first source screenshot. Line_Total is the stored source measure, not recomputed from Price*Quantity: five original rows are inconsistent. Preserve them to reproduce source grouped/overall totals rather than silently change the data. Exact original screenshots remain in the blog. The last screenshot contains four wrong Daily_Total entries (quantities instead of amount totals); current outputs are calculated from actual Line_Total values and match independent Decimal-based expectations.

All six original SQL examples remain. Correct Quanitity to Quantity, replace duplicate Sales_Total count aliases by Line_Count, remove a trailing comma, retain Sales_details_ID as in source result figures and make final display ordering explicit. The CTE/join and window-sum query use the same explicit projection and are compared row for row. Window COUNT(Line_Total) counts nonnull amounts, not necessarily all rows in a nullable dataset; regressions exercise that distinction.

OVER(PARTITION BY Sales_Id), OVER(PARTITION BY Sales_Date) and OVER() compute whole-sale, whole-day and whole-table totals. No window ORDER BY is added, so these are not cumulative running totals or moving averages. Introductory source promises and SQL function-list simplifications are preserved in the article but scoped in the private review; no additional absent tutorial is invented. No universal performance advantage is claimed.

Line_Total is nullable in edge tests: SUM is NULL for all-null groups and COUNT(Line_Total) is zero. Empty grouped/window SELECTs return no rows. Final cell checks every full result, 32 additional seeded synthetic fixtures and unchanged source data, then closes connections. Git notebook outputs/counts remain empty/null.


In [ ]:
import sqlite3
from decimal import Decimal
from IPython.core.magic import register_cell_magic

assert sqlite3.sqlite_version_info >= (3, 25, 0), 'SQLite 3.25+ is required'
connection = sqlite3.connect(':memory:')
SCHEMA = """
CREATE TABLE Sales_Details (
    Sales_details_ID INTEGER NOT NULL PRIMARY KEY,
    Sales_Id INTEGER NOT NULL,
    Sales_Date TEXT NOT NULL,
    Item INTEGER NOT NULL,
    Price REAL NOT NULL,
    Quantity INTEGER NOT NULL,
    Line_Total REAL,
    CHECK (date(Sales_Date) IS NOT NULL AND date(Sales_Date) = Sales_Date)
);
"""
connection.executescript(SCHEMA)
# All ten rows recovered from the original input figure; keep stored Line_Total.
sales_fixture = [
    (1, 14, '2023-04-11', 15, 5.60, 2, 84.00),
    (2, 12, '2023-04-12', 56, 4.98, 4, 63.90),
    (3, 8, '2023-04-13', 65, 12.78, 5, 830.70),
    (4, 12, '2023-04-12', 54, 16.78, 6, 100.68),
    (5, 2, '2023-04-16', 54, 50.00, 9, 459.00),
    (6, 8, '2023-04-01', 18, 15.00, 12, 180.00),
    (7, 5, '2023-04-02', 19, 64.62, 16, 775.44),
    (8, 20, '2023-04-03', 25, 15.87, 4, 63.48),
    (9, 17, '2023-04-10', 27, 41.56, 3, 124.68),
    (10, 9, '2023-04-12', 15, 32.54, 18, 585.72),
]
connection.executemany('INSERT INTO Sales_Details VALUES (?, ?, ?, ?, ?, ?, ?)', sales_fixture)
query_results = {}
query_texts = {}

def print_result(columns, rows):
    text_rows = [[str(value) if value is not None else 'NULL' for value in row] for row in rows]
    widths = [max([len(column)] + [len(row[i]) for row in text_rows]) for i, column in enumerate(columns)]
    def format_row(row):
        return ' | '.join(value.ljust(width) for value, width in zip(row, widths)).rstrip()
    print(format_row(columns))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(format_row(row))
    print('Rows:', len(rows))

@register_cell_magic
def sql(line, cell):
    key = line.strip()
    cursor = connection.execute(cell)
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    query_texts[key] = cell.strip()
    query_results[key] = (columns, rows)
    print_result(columns, rows)


print('SQLite:', sqlite3.sqlite_version)
print('Recovered source input: 10 sales-detail rows, 8 sale IDs, 8 dates.')
print('SQL magic ready. All data stays in-memory; no files, server, keys or downloads.')


## Query 1


In [ ]:
%%sql example-0
SELECT Sales_Id, SUM(Line_Total) AS Total
FROM Sales_Details
GROUP BY Sales_Id
ORDER BY Sales_Id;


## Query 2


In [ ]:
%%sql example-1
WITH CTE AS (
    SELECT Sales_Id, SUM(Line_Total) AS Total
    FROM Sales_Details
    GROUP BY Sales_Id
)
SELECT B.Sales_details_ID, B.Sales_Id, B.Sales_Date, B.Item,
       B.Price, B.Quantity, B.Line_Total, A.Total AS Sales_Total
FROM CTE AS A
INNER JOIN Sales_Details AS B ON A.Sales_Id = B.Sales_Id
ORDER BY B.Sales_details_ID;


## Query 3


In [ ]:
%%sql example-2
SELECT Sales_details_ID, Sales_Id, Sales_Date, Item, Price, Quantity, Line_Total,
       SUM(Line_Total) OVER (PARTITION BY Sales_Id) AS Sales_Total
FROM Sales_Details
ORDER BY Sales_details_ID;


## Query 4


In [ ]:
%%sql example-3
SELECT Sales_details_ID, Sales_Id, Sales_Date, Item, Price, Quantity, Line_Total,
       COUNT(Line_Total) OVER (PARTITION BY Sales_Id) AS Line_Count,
       SUM(Line_Total) OVER (PARTITION BY Sales_Id) AS Sales_Total
FROM Sales_Details
ORDER BY Sales_details_ID;


## Query 5


In [ ]:
%%sql example-4
SELECT Sales_details_ID, Sales_Id, Sales_Date, Item, Price, Quantity, Line_Total,
       COUNT(Line_Total) OVER (PARTITION BY Sales_Id) AS Line_Count,
       SUM(Line_Total) OVER (PARTITION BY Sales_Id) AS Sales_Total,
       SUM(Line_Total) OVER () AS Total
FROM Sales_Details
ORDER BY Sales_details_ID;


## Query 6


In [ ]:
%%sql example-5
SELECT Sales_details_ID, Sales_Id, Sales_Date, Item, Price, Quantity, Line_Total,
       COUNT(Line_Total) OVER (PARTITION BY Sales_Id) AS Line_Count,
       SUM(Line_Total) OVER (PARTITION BY Sales_Id) AS Sales_Total,
       SUM(Line_Total) OVER (PARTITION BY Sales_Date) AS Daily_Total,
       SUM(Line_Total) OVER () AS Total
FROM Sales_Details
ORDER BY Sales_details_ID;


## Regression checks


In [ ]:
import math
import random
from collections import defaultdict
from datetime import date, timedelta

def compare(actual, expected):
    assert len(actual) == len(expected)
    for left, right in zip(actual, expected):
        if isinstance(right, (list, tuple)):
            compare(left, right)
        elif right is None:
            assert left is None
        elif isinstance(right, float):
            assert math.isclose(left, right, rel_tol=1e-12, abs_tol=1e-12), (left, right)
        else:
            assert left == right, (left, right)

def expected_results(rows):
    sales, dates = defaultdict(list), defaultdict(list)
    for row in rows:
        sales[row[1]].append(row[6])
        dates[row[2]].append(row[6])
    def total(values):
        observed = [Decimal(str(value)) for value in values if value is not None]
        return float(sum(observed)) if observed else None
    sale_totals = {key: total(values) for key, values in sales.items()}
    daily_totals = {key: total(values) for key, values in dates.items()}
    counts = {key: sum(value is not None for value in values) for key, values in sales.items()}
    grand = total([row[6] for row in rows])
    ordered = sorted(rows)
    with_sum = [(*row, sale_totals[row[1]]) for row in ordered]
    with_count = [(*row, counts[row[1]], sale_totals[row[1]]) for row in ordered]
    return [
        [(key, sale_totals[key]) for key in sorted(sales)],
        with_sum, with_sum, with_count,
        [(*row, grand) for row in with_count],
        [(*row, counts[row[1]], sale_totals[row[1]], daily_totals[row[2]], grand) for row in ordered],
    ]

baseline = expected_results(sales_fixture)
for i in range(6):
    columns, rows = query_results['example-' + str(i)]
    assert len(columns) == len(set(columns))
    compare(rows, baseline[i])
compare(query_results['example-1'][1], query_results['example-2'][1])
assert len(query_results['example-0'][1]) == 8
assert all(len(query_results['example-' + str(i)][1]) == 10 for i in range(1, 6))
assert all(math.isclose(row[-1], 3267.6) for row in query_results['example-5'][1])
daily = {row[2]: row[-2] for row in query_results['example-5'][1]}
compare([daily[day] for day in ['2023-04-01', '2023-04-02', '2023-04-03', '2023-04-10', '2023-04-12']],
        [180.0, 775.44, 63.48, 124.68, 750.3])

def check_fixture(rows):
    db = sqlite3.connect(':memory:')
    db.executescript(SCHEMA)
    db.executemany('INSERT INTO Sales_Details VALUES (?, ?, ?, ?, ?, ?, ?)', rows)
    expected = expected_results(rows)
    for i in range(6):
        compare(db.execute(query_texts['example-' + str(i)]).fetchall(), expected[i])
    db.close()

check_fixture([])
check_fixture(list(reversed(sales_fixture)))
check_fixture([(1, 1, '2023-01-01', 1, 0.0, 1, 0.0)])
check_fixture([(1, 1, '2023-01-01', 1, 0.0, 1, None)])
nullable = [(1, 1, '2023-01-01', 1, 0.0, 1, None),
            (2, 1, '2023-01-02', 2, 5.0, 1, 5.0),
            (3, 2, '2023-01-02', 3, 0.0, 1, None)]
check_fixture(nullable)
assert expected_results(nullable)[3][0][-2] == 1  # COUNT(Line_Total), not COUNT(*).
check_fixture([(1, 1, '2023-01-01', 1, 3.0, 1, 3.0),
               (2, 1, '2023-01-02', 2, 3.0, 1, -3.0)])
rng = random.Random(42)
for case in range(32):
    rows = [(i + 1, rng.randrange(1, 6), (date(2023, 1, 1) + timedelta(days=rng.randrange(4))).isoformat(),
             i + 1, 1.0, 1, rng.choice([None, 0.0, rng.randrange(-1000, 10000) / 100]))
            for i in range(rng.randrange(21))]
    rng.shuffle(rows)
    check_fixture(rows)

# Record, do not overwrite, five source Price*Quantity versus Line_Total mismatches.
mismatches = [row[0] for row in sales_fixture if Decimal(str(row[4])) * row[5] != Decimal(str(row[6]))]
assert mismatches == [1, 2, 3, 5, 7]
assert connection.execute('SELECT * FROM Sales_Details ORDER BY Sales_details_ID').fetchall() == sales_fixture
connection.close()
print('All six SQL statements and complete independent Decimal-result comparisons passed.')
print('Grouped output: 8 rows; joined and window outputs: all 10 source rows. Grand total: 3267.6.')
print('Daily totals checked, including 180.0, 775.44, 63.48, 124.68 and 750.3.')
print('Tested empty, reversed input, singleton, null/all-null, zero/negative amounts and 32 seeded fixtures.')
print('Original input amounts stayed unchanged; all databases stayed in-memory.')
